# 01 · 解析 PubMed 数据（nbib）

本 notebook 的目标：把 PubMed 导出的原始 `.nbib` 变成干净的、可供分析的中间表。

- 直接解析 `data/raw/*.nbib`（**不经过 Zotero**）
- 合并所有年份、按 `PMID` 去重
- 输出：
  - `data/interim/articles.csv` —— 一篇文章一行
  - `data/interim/authors.csv` —— 一行一个作者（含 ORCID、单位）


## 第一步：读取数据

找到 `data/raw/` 下的所有 `.nbib` 原始文件。

In [1]:
import os
import glob

# --- 原始数据目录 ---
# notebook 位于 notebooks/，数据在上一级的 data/raw/
raw_dir = os.path.join(os.getcwd(), '..', 'data', 'raw')

# --- 找到所有 nbib 文件 ---
nbib_files = sorted(glob.glob(os.path.join(raw_dir, 'pubmed-XJTU-*.nbib')))

print(f"在 {raw_dir} 找到 {len(nbib_files)} 个 nbib 文件：")
for f in nbib_files:
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f"  {os.path.basename(f):28s}  {size_mb:6.1f} MB")


在 D:\MEDscience_map_of_XJTU.wt\analysis\notebooks\..\data\raw 找到 6 个 nbib 文件：
  pubmed-XJTU-2021.nbib           21.8 MB
  pubmed-XJTU-2022.nbib           27.0 MB
  pubmed-XJTU-2023.nbib           25.2 MB
  pubmed-XJTU-2024.nbib           29.3 MB
  pubmed-XJTU-2025.nbib           36.4 MB
  pubmed-XJTU-2026.nbib           29.0 MB


## 第二步：检查原始数据有哪些字段

`nbib`（MEDLINE 格式）是 `标签 - 值` 的纯文本，字段是隐藏的。
先把一个文件里出现的所有字段标签列出来，看看有哪些可用元数据。

In [2]:
from collections import Counter

# 拿第一个文件做代表，扫描它出现的所有字段标签
sample_file = nbib_files[0]

tag_counts = Counter()
tag_sample = {}
with open(sample_file, encoding='utf-8') as f:
    for line in f:
        # 字段行形如 'TAG  - value'；续行以空格开头，跳过
        if len(line) > 6 and line[0] != ' ' and '-' in line[:6]:
            tag = line[:4].strip()
            value = line[6:].strip()
            tag_counts[tag] += 1
            tag_sample.setdefault(tag, value[:45])

print(f"{os.path.basename(sample_file)} 中共出现 {len(tag_counts)} 种字段：")
print()
print(f"{'标签':<6}{'次数':>7}   示例")
for tag, cnt in tag_counts.most_common():
    print(f"{tag:<6}{cnt:>7}   {tag_sample.get(tag, '')}")

pubmed-XJTU-2021.nbib 中共出现 61 种字段：

标签         次数   示例
AD      44853   Key Laboratory of Shaanxi Province for Cranio
FAU     38604   Sun, Huiling
AU      38604   Sun H
MH      31797   Animals
PHST    24952   2021/02/25 00:00 [received]
OT      17769   Exendin ((9-39))
IS      10681   1879-0712 (Electronic)
AID      8598   S0014-2999(21)00225-9 [pii]
RN       7911   0 (Appetite Depressants)
LID      6930   S0014-2999(21)00225-9 [pii]
PT       6866   Journal Article
LA       4572   eng
PMID     4563   33823184
OWN      4563   NLM
STAT     4563   MEDLINE
LR       4563   20210518
DP       4563   2021 Jun 15
TI       4563   Melanocortin receptor-4 mediates the anorecti
PL       4563   Netherlands
TA       4563   Eur J Pharmacol
JT       4563   European journal of pharmacology
JID      4563   1254354
EDAT     4563   2021/04/07 06:00
MHDA     4563   2021/05/19 06:00
CRDT     4563   2021/04/06 20:08
PST      4563   ppublish
SO       4563   Eur J Pharmacol. 2021 Jun 15;901:174072. doi:
VI      

## 字段说明与建议

### 为什么每种标签的出现次数不一样？

有两个原因（以 2021 年 4,563 篇为例）：

**原因 1 · 有些标签一篇论文会重复多次**
- `AD`（单位）出现 44,853 次 —— 一篇文章有几个作者就有几个单位
- `AU` / `FAU`（作者）38,604 次 —— 一篇多作者就多行
- `MH`（主题词）31,797 次 —— 一篇能标十几个 MeSH 词
- 这类「重复字段」的总次数会远超论文数。

**原因 2 · 有些标签不是每篇都有（可选项）**
- `MH` 只覆盖 64.9% —— 部分记录还没被 MEDLINE 标引
- `AB`（摘要）覆盖 96.6% —— 157 篇没有摘要
- `AUID`（ORCID）覆盖 43.1% —— 只有注册 ORCID 的作者才有
- `GR`（基金）21.3%、`PMC` 51.5%、`OT`（作者关键词）78%
- `VI / IP / PG`（卷期页）—— 电子优先发表的文章还没分卷期页

> 结论：`PMID` 这类「单值必备字段」每篇都有（= 4,563）；`AU`/`MH` 这类「重复字段」一篇多行；`AB`/`AUID` 这类「可选项」不是每篇都有。

### 🟢 核心字段（最终保留这 12 个）

| 标签 | 中文含义 | 示例 | 覆盖 |
|---|---|---|---|
| `PMID` | PubMed 编号 | `33823184` | 100% |
| `TI` | 标题 | `Melanocortin receptor-4…` | 100% |
| `AB` | 摘要 | `Glucagon-like peptide-2…` | 96.6% |
| `FAU` | 作者全名 | `Sun, Huiling` | 100% |
| `AD` | 作者单位 | `Key Laboratory of Shaanxi…` | 100% |
| `AUID` | 作者标识符（ORCID） | `ORCID: 0000-0002-…` | 43.1% |
| `MH` | MeSH 主题词（受控词） | `Animals` | 64.9% |
| `OT` | 作者关键词（自由词） | `Exendin` | 78.0% |
| `PT` | 出版类型 | `Journal Article` | 100% |
| `DP` | 出版日期 | `2021 Jun 15` | 100% |
| `JT` | 期刊全名 | `European journal of pharmacology` | 100% |
| `LID` | 位置标识（含 DOI） | `…[doi]` | 99.3% |

> 注：DOI 也可能出现在 `AID`（文章标识符）里，解析时作为 `LID` 的备用来源一起找。

### 🟡 可选字段

| 标签 | 中文含义 | 示例 | 覆盖 | 建议 |
|---|---|---|---|---|
| `AU` | 作者缩写名 | `Sun H` | 100% | △ 备选（不如 FAU 全） |
| `AID` | 文章标识符（含 DOI/PII） | `S0014-…[pii]` | 99.4% | △ DOI 备用来源 |
| `LA` | 语言 | `eng` | 100% | △ 筛英文 |
| `IS` | ISSN | `1879-0712 (Electronic)` | 100% | ✗ |
| `VI` | 卷 | `901` | 99.7% | △ |
| `IP` | 期 | `5` | 71.2% | △ |
| `PG` | 页码 | `174072` | 91.3% | △ |
| `DEP` | 电子出版日期 | `20210403` | 87.4% | △ |
| `SB` | 期刊子集 | `IM` | 76.4% | △ |
| `GR` | 基金资助 | `81701869/National…` | 21.3% | △ 基金分析 |
| `RN` | 化学物质登记号 | `0 (Appetite Depressants)` | 43.5% | △ |
| `PMC` | PMC 编号 | `PMC7981806` | 51.5% | △ 开放获取 |
| `SI` | 次要来源 ID（试验注册号） | `ClinicalTrials.gov/NCT…` | 1.3% | △ |
| `COIS` | 利益冲突声明 | `Declaration of Competing…` | 52.9% | △ |
| `CN` | 团体作者 | `STEP Study Group` | 0.6% | △ |
| `FIR` | 研究者全名（团体） | `Cai, Jun` | 0.2% | △ |
| `PHST` | 出版流程状态日期 | `2021/02/25 [received]` | 100% | ✗ |
| `PST` | 出版状态 | `ppublish` | 100% | ✗ |
| `EDAT` | 入库日期 | `2021/04/07` | 100% | ✗ |

### ⚪ 一般忽略

`OWN`(记录所有者) · `STAT`(记录状态) · `LR`(最后修订) · `PL`(出版国家) · `JID`(期刊ID) · `MHDA`(MeSH标引日期) · `CRDT`(创建日期) · `SO`(来源引用串) · `OTO`(OT所有者) · `DCOM`(完成日期) · `CI`(版权) · `PMCR`(PMC发布日期) · `IR`(研究者缩写) · `IRAD`(研究者单位) · `MID`(稿号) · `CIN`(被评论) · `CON`(评论对象) · `EIN`(被勘误) · `EFR`(勘误对象) · `RIN`(撤稿来源) · `ROF`(撤稿对象) · `ECI`(关注声明) · `RRI`(再撤稿相关) · `UOF`(更新对象) · `TT`(原文标题) · `OAB`(其他摘要) · `OABL`(其他摘要语言) · `PS`(作为主题的人名) · `FPS`(作为主题的人名全名)

### ✅ 结论

解析器保留这 **12 个核心字段**：`PMID`、`TI`、`AB`、`FAU`、`AD`、`AUID`、`MH`、`OT`、`PT`、`DP`、`JT`、`LID`。


## 第三步：解析单条记录

MEDLINE（`.nbib`）的每条记录都是「标签 - 值」结构。解析时要先弄清两件事：

**1. 续行**：值太长时会折行，续行以空格开头，要拼回上一行。

**2. 作者块**：作者信息是成组出现的，顺序很关键——

```
FAU - Sun, Huiling       <- 作者全名（一个作者块从 FAU 开始）
AU  - Sun H              <- 缩写名
AUID- ORCID: 0000-...    <- 该作者的 ORCID（可选）
AD  - Key Laboratory...  <- 该作者的单位（可有多条）
FAU - Meng, Kai          <- 下一个作者
...
```

所以必须**按顺序**读：遇到 `FAU` 就开一个新作者，后面的 `AUID` / `AD` 都归到这个作者名下，直到下一个 `FAU`。
这样「作者 ↔ 单位 ↔ ORCID」才能对应起来——这正是上一版做不到、导致作者消歧失败的地方。


In [3]:
def parse_record(text):
    '''把一条 MEDLINE 记录解析成 (普通字段, 作者块)。

    普通字段: {标签: [值, ...]}，如 fields['MH'] 是该文所有 MeSH 词。
    作者块:   [{'name': 全名, 'orcid': ORCID 或 None, 'affiliations': [单位, ...]}, ...]
    '''
    fields = {}
    authors = []
    cur_tag = None      # 当前正在读的标签（用于拼接续行）
    cur_author = None   # 当前作者块

    def add(tag, value):
        fields.setdefault(tag, []).append(value)

    for line in text.split('\n'):
        if not line.strip():
            continue

        if line[0] == ' ':                       # 续行：拼到上一个值后面
            value = line.strip()
            if cur_tag == 'FAU' and cur_author is not None:
                cur_author['name'] += ' ' + value
            elif cur_tag == 'AD' and cur_author is not None and cur_author['affiliations']:
                cur_author['affiliations'][-1] += ' ' + value
            elif cur_tag is not None and fields.get(cur_tag):
                fields[cur_tag][-1] += ' ' + value

        else:                                    # 新字段行，形如 'FAU - Sun, Huiling'
            tag = line[:4].strip()
            value = line[6:].strip()
            cur_tag = tag
            add(tag, value)

            if tag == 'FAU':                     # 开一个新作者块
                cur_author = {'name': value, 'orcid': None, 'affiliations': []}
                authors.append(cur_author)
            elif tag == 'AUID' and cur_author is not None and cur_author['orcid'] is None:
                cur_author['orcid'] = value      # AUID 属于当前作者
            elif tag == 'AD' and cur_author is not None:
                cur_author['affiliations'].append(value)   # AD 属于当前作者

    return fields, authors


先拿第一条记录试一下，确认解出来的结构对不对。

In [4]:
sample_text = open(nbib_files[0], encoding='utf-8').read().split('\n\n')[0]
sample_fields, sample_authors = parse_record(sample_text)

print("普通字段示例：")
for tag in ['PMID', 'TI', 'DP', 'JT', 'PT', 'LID', 'MH', 'OT']:
    print(f"  {tag:<5}: {sample_fields.get(tag)}")

print("\n作者块（作者 ↔ 单位 ↔ ORCID 已对应）：")
for a in sample_authors:
    print(f"  {a['name']:<24} ORCID={a['orcid']}")
    for ad in a['affiliations']:
        print(f"      └ {ad[:70]}")


普通字段示例：
  PMID : ['33823184']
  TI   : ['Melanocortin receptor-4 mediates the anorectic effect induced by the nucleus tractus solitarius injection of glucagon-like Peptide-2 in fasted rats.']
  DP   : ['2021 Jun 15']
  JT   : ['European journal of pharmacology']
  PT   : ['Journal Article']
  LID  : ['S0014-2999(21)00225-9 [pii]', '10.1016/j.ejphar.2021.174072 [doi]']
  MH   : ['Animals', 'Appetite Depressants/*pharmacology', 'Eating/drug effects', 'Fasting', 'Glucagon-Like Peptide 2/administration & dosage/antagonists & inhibitors/*pharmacology', 'Male', 'Melanocyte-Stimulating Hormones/pharmacology', 'Microinjections', 'Peptide Fragments/pharmacology', 'Rats', 'Rats, Sprague-Dawley', 'Receptor, Melanocortin, Type 4/*drug effects', '*Solitary Nucleus']
  OT   : ['Exendin ((9-39))', 'Food intake', 'Neurotransmitter', 'Nucleus tractus solitarius (NTS)', 'SHU 9119']

作者块（作者 ↔ 单位 ↔ ORCID 已对应）：
  Sun, Huiling             ORCID=None
      └ Key Laboratory of Shaanxi Province for Craniofacia

## 第四步：把一条记录整理成一行

接下来把上面的「标签字典」压成表格的一行（宽表）。规则：

| 目标列 | 来源 | 处理方式 |
|---|---|---|
| `pmid` / `title` / `abstract` | `PMID` / `TI` / `AB` | 单值 |
| `authors` | `FAU` | 多值，按 `||` 拼接 |
| `first_author` / `last_author` / `n_authors` | `FAU` | 从作者块派生 |
| `orcids` | `AUID` | 多值，去重 |
| `affiliations` | `AD` | 多值，去重 |
| `mesh` / `keywords` / `pubtypes` | `MH` / `OT` / `PT` | 多值 |
| `journal` | `JT`（备用 `TA`） | 单值 |
| `pubdate` | `DP` | 单值（期刊卷期日期） |
| `year` | `DEP`（备用 `DP`） | 提取 4 位年份 |
| `doi` | `LID` / `AID` | 取以 `[doi]` 结尾的那条 |
| `language` / `volume` / `issue` / `pages` | `LA` / `VI` / `IP` / `PG` | 顺带保留 |

> **多值列的分隔符用 `||`**：它不会出现在作者名、单位、MeSH、关键词里（只会出现在摘要正文中，而摘要始终当单值处理、不会被拆开），所以之后按它拆回来是安全的。


In [5]:
import re

import pandas as pd

# 多值字段的分隔符（详见上面说明）
MULTI_SEP = '||'


def first(fields, tag, default=''):
    '''取某个单值字段的第一个值。'''
    values = fields.get(tag)
    return values[0] if values else default


def joined(fields, tag):
    '''把某个多值字段拼成一个字符串。'''
    return MULTI_SEP.join(fields.get(tag, []))


def unique(seq):
    '''去重，同时保持原来的顺序。'''
    seen, out = set(), []
    for x in seq:
        if x and x not in seen:
            seen.add(x)
            out.append(x)
    return out


def find_doi(fields):
    '''从 LID / AID 里找出 DOI（值以 [doi] 结尾）。'''
    for tag in ('LID', 'AID'):
        for value in fields.get(tag, []):
            if value.endswith('[doi]'):
                return value[:-len('[doi]')].strip()
    return ''


def find_year(fields):
    '''年份：优先用电子出版日期 DEP，没有再用出版日期 DP。'''
    for tag in ('DEP', 'DP'):
        text = first(fields, tag)
        if len(text) >= 4 and text[:4].isdigit():
            return int(text[:4])
    return None


In [6]:
def record_to_row(text):
    '''一条记录 -> (宽表的一行 dict, 作者块列表)。'''
    fields, authors = parse_record(text)
    if 'PMID' not in fields:
        return None, None

    row = {
        'pmid':         first(fields, 'PMID'),
        'doi':          find_doi(fields),
        'title':        first(fields, 'TI'),
        'abstract':     first(fields, 'AB'),
        # --- 作者 ---
        'authors':      MULTI_SEP.join(a['name'] for a in authors),
        'first_author': authors[0]['name'] if authors else '',
        'last_author':  authors[-1]['name'] if authors else '',
        'n_authors':    len(authors),
        'orcids':       MULTI_SEP.join(unique(a['orcid'] for a in authors if a['orcid'])),
        # --- 单位（所有作者的，去重）---
        'affiliations': MULTI_SEP.join(unique(ad for a in authors for ad in a['affiliations'])),
        # --- 主题词 / 关键词 / 出版类型 ---
        'mesh':         joined(fields, 'MH'),
        'keywords':     joined(fields, 'OT'),
        'pubtypes':     joined(fields, 'PT'),
        # --- 出处 ---
        'journal':      first(fields, 'JT') or first(fields, 'TA'),
        'pubdate':      first(fields, 'DP'),
        'year':         find_year(fields),
        'language':     joined(fields, 'LA'),
        'volume':       first(fields, 'VI'),
        'issue':        first(fields, 'IP'),
        'pages':        first(fields, 'PG'),
    }
    return row, authors


## 第五步：批量解析全部文件并合并

遍历 `data/raw/` 下的每个 `.nbib`，逐条解析、攒成两张表：

- **`df`（文章宽表）**：一篇文章一行。
- **`df_authors`（作者长表）**：一行 = 一篇文章里的一个作者，保留顺序、ORCID、单位。
  这张表是专门为**作者消歧**准备的（见 `docs/审查问题记录.md` #02）。

> ⚠️ 关于年份范围：研究日志写的是 2021–2025，但 `data/raw/` 里目前还有一个 **2026** 年的文件，
> 上面的 `glob` 会把它一起读进来。本 notebook 默认保留全部年份、交给下一步再筛；
> 若只想分析 2021–2025，可在保存前加一句 `df = df[df['year'].between(2021, 2025)]`。


In [7]:
rows = []
author_rows = []          # 作者长表

for file in nbib_files:
    text = open(file, encoding='utf-8').read()
    records = [r for r in text.split('\n\n') if r.strip()]
    n_ok = 0
    for record in records:
        row, authors = record_to_row(record)
        if row is None:
            continue
        row['source_file'] = os.path.basename(file)
        rows.append(row)
        n_ok += 1
        for seq, a in enumerate(authors, start=1):
            author_rows.append({
                'pmid':         row['pmid'],
                'seq':          seq,
                'full_name':    a['name'],
                'orcid':        a['orcid'] or '',
                'affiliations': MULTI_SEP.join(a['affiliations']),
            })
    print(f"{os.path.basename(file):28s} 解析 {n_ok:>5} / {len(records):>5} 条")

df = pd.DataFrame(rows)
df_authors = pd.DataFrame(author_rows)

print(f"\n合并完成：{len(df)} 条记录（去重前）")
print(f"作者长表：{len(df_authors)} 行")


pubmed-XJTU-2021.nbib        解析  4563 /  4563 条


pubmed-XJTU-2022.nbib        解析  5527 /  5527 条


pubmed-XJTU-2023.nbib        解析  5081 /  5081 条


pubmed-XJTU-2024.nbib        解析  5663 /  5663 条


pubmed-XJTU-2025.nbib        解析  6816 /  6816 条


pubmed-XJTU-2026.nbib        解析  5223 /  5223 条



合并完成：32873 条记录（去重前）
作者长表：303351 行


## 第六步：按 PMID 去重

逐年检索会带来**跨年重复**：同一篇论文可能因电子版 / 纸质版日期不同，落进相邻两年的结果里。
`PMID` 是 PubMed 的全局唯一编号，用它去重最可靠。这里**保留最早出现的那条**（文件按年份排序）。


In [8]:
before = len(df)

duplicated = df[df['pmid'].duplicated(keep=False)].sort_values('pmid')
print(f"跨年重复的 PMID：{duplicated['pmid'].nunique()} 个，对应 {len(duplicated)} 条记录")

df = df.drop_duplicates('pmid', keep='first').reset_index(drop=True)
print(f"按 PMID 去重：{before} -> {len(df)}（删掉 {before - len(df)} 条）")

# 作者长表同步：只保留留下来的 PMID，并按 (pmid, seq) 去重
kept = set(df['pmid'])
df_authors = (df_authors[df_authors['pmid'].isin(kept)]
              .drop_duplicates(['pmid', 'seq'], keep='first')
              .reset_index(drop=True))
print(f"作者长表同步过滤后：{len(df_authors)} 行")


跨年重复的 PMID：2966 个，对应 5932 条记录
按 PMID 去重：32873 -> 29907（删掉 2966 条）
作者长表同步过滤后：275392 行


## 第七步：质量检查

看一下每个字段的覆盖率、年份分布，并随机拆一行出来确认多值字段能还原。


In [9]:
def nonempty_ratio(s):
    '''字段的非空比例。'''
    if s.dtype == object:
        return (s.fillna('').astype(str).str.strip() != '').mean()
    return s.notna().mean()


coverage = pd.DataFrame({
    '类型': [str(df[c].dtype) for c in df.columns],
    '覆盖率': [f"{nonempty_ratio(df[c]) * 100:.1f}%" for c in df.columns],
}, index=df.columns)
coverage


,类型,覆盖率
pmid,object,100.0%
doi,object,99.6%
title,object,100.0%
abstract,object,96.4%
authors,object,100.0%
first_author,object,100.0%
last_author,object,100.0%
n_authors,int64,100.0%
orcids,object,44.7%
affiliations,object,100.0%


In [10]:
print("按电子出版年份（DEP，缺失时用 DP）统计：")
year_counts = df['year'].dropna().astype(int).value_counts().sort_index()
for year, count in year_counts.items():
    print(f"  {year}: {count:>5} 篇")
print(f"\n合计 {int(year_counts.sum())} 篇")

print("\n看一行拆开后的样子：")
i = 0
print("  标题   :", df.loc[i, 'title'][:60])
print("  作者   :", df.loc[i, 'authors'].split(MULTI_SEP)[:3], "...")
print("  MeSH   :", df.loc[i, 'mesh'].split(MULTI_SEP)[:3], "...")
print("  关键词 :", df.loc[i, 'keywords'].split(MULTI_SEP)[:3], "...")
print("  单位   :", df.loc[i, 'affiliations'].split(MULTI_SEP)[0][:60], "...")

pd.set_option('display.max_colwidth', 40)
df[['pmid', 'year', 'first_author', 'n_authors', 'journal', 'doi']].head()


按电子出版年份（DEP，缺失时用 DP）统计：
  2018:     1 篇
  2019:    10 篇
  2020:   517 篇
  2021:  4021 篇
  2022:  4950 篇
  2023:  4527 篇
  2024:  5098 篇
  2025:  6170 篇
  2026:  4613 篇

合计 29907 篇

看一行拆开后的样子：
  标题   : Melanocortin receptor-4 mediates the anorectic effect induce
  作者   : ['Sun, Huiling', 'Meng, Kai', 'Hou, Lin'] ...
  MeSH   : ['Animals', 'Appetite Depressants/*pharmacology', 'Eating/drug effects'] ...
  关键词 : ['Exendin ((9-39))', 'Food intake', 'Neurotransmitter'] ...
  单位   : Key Laboratory of Shaanxi Province for Craniofacial Precisio ...


,pmid,year,first_author,n_authors,journal,doi
0,33823184,2021,"Sun, Huiling",5,European journal of pharmacology,10.1016/j.ejphar.2021.174072
1,33812273,2021,"Kou, Xing",3,Ultrasonics,10.1016/j.ultras.2021.106426
2,33762094,2021,"Feng, Yong",6,ISA transactions,10.1016/j.isatra.2021.03.013
3,33760976,2021,"Zhao, Jia-Min",9,International journal of legal medicine,10.1007/s00414-021-02559-2
4,33744670,2021,"Cao, Shuai",6,Biosensors & bioelectronics,10.1016/j.bios.2021.113164


## 第八步：作者身份归一（消歧）

「作者块 ↔ 作者条目」的对应已经可靠，但**同一个全名不代表同一个人**（例如两位 `Wang, Kai` 在不同单位）。
所以给每个作者条目分配一个稳定的 `author_key`。规则**精度优先**（宁可把同一人拆开，也不要把不同人并起来）：

1. 有 **ORCID** → `orcid:0000-...`（全球唯一作者号，最可靠）。
2. 没有 ORCID → `name:<规范化全名> @ <规范化单位>`（取该作者的第一个单位）。
   - 同名**不同单位** → 不同 key（不会被错误合并，正好修 #02）；
   - 同名**同单位** → 同一 key。

> 局限（留给 02 去检查和改进）：改名、跨单位调动、同名同单位仍会误判；ORCID 覆盖率约 44.7%。

In [11]:
# =====================================================================
# 作者身份归一：给每个作者条目分配一个稳定的 author_key
# =====================================================================
def normalize_name(name):
    '''全名规范化：小写、去标点、压缩空格。'Sun, Huiling' -> 'sun huiling'。'''
    s = name.lower()
    s = re.sub(r'[.,;:()\[\]{}"\'/\-]+', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()


def normalize_affiliation(aff):
    '''单位规范化：小写、去标点、压缩空格。'''
    s = aff.lower()
    s = re.sub(r'[^a-z0-9\u4e00-\u9fff]+', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()


def make_author_key(name, orcid, affiliations):
    '''ORCID 优先；否则用「全名 @ 第一个单位」。'''
    if orcid:
        return 'orcid:' + orcid.replace('ORCID:', '').strip()
    first_aff = affiliations[0] if affiliations else ''
    return 'name:' + normalize_name(name) + ' @ ' + normalize_affiliation(first_aff)


df_authors['author_key'] = df_authors.apply(
    lambda r: make_author_key(r['full_name'], r['orcid'], r['affiliations'].split(MULTI_SEP)),
    axis=1,
)

# 把作者键按顺序拼回文章表，下一步分析就不用再 join 了
keys_by_pmid = df_authors.groupby('pmid')['author_key'].apply(lambda s: MULTI_SEP.join(s))
df['authors_key'] = df['pmid'].map(keys_by_pmid).fillna('')

has_orcid = df_authors['orcid'] != ''
print(f"作者条目：{len(df_authors)}")
print(f"  有 ORCID（key 用 orcid）    : {has_orcid.sum():>7}  ({has_orcid.mean() * 100:.1f}%)")
print(f"  无 ORCID（key 用 全名+单位）: {(~has_orcid).sum():>7}  ({(~has_orcid).mean() * 100:.1f}%)")
print(f"不同全名：{df_authors['full_name'].nunique()}    不同 author_key：{df_authors['author_key'].nunique()}")

作者条目：275392
  有 ORCID（key 用 orcid）    :   36273  (13.2%)
  无 ORCID（key 用 全名+单位）:  239119  (86.8%)
不同全名：87724    不同 author_key：222071


In [12]:
# 例子：同名的两位 Wang, Kai 被拆成不同的 key
demo = df_authors[df_authors['pmid'] == '40477158']
for _, r in demo[demo['full_name'] == 'Wang, Kai'].iterrows():
    print(f"  {r['full_name']}  序{r['seq']}  ->  {r['author_key'][:95]}")

# 高产作者 Top10（同一篇文章里同一作者只算一次）
top = (df_authors.drop_duplicates(['pmid', 'author_key'])
       .groupby('author_key').size().sort_values(ascending=False).head(10))
name_of = df_authors.drop_duplicates('author_key').set_index('author_key')['full_name']
print("\n按 author_key 统计的高产作者 Top10：")
for key, n in top.items():
    print(f"  {n:>4} 篇  {name_of.get(key, '?'):<20} {key.split('@')[-1].strip()[:42]}")

  Wang, Kai  序2  ->  name:wang kai @ jiangsu provincial key laboratory of geriatrics department of geriatrics the fi
  Wang, Kai  序4  ->  name:wang kai @ department of medical engineering and technology xinjiang medical university ur



按 author_key 统计的高产作者 Top10：
    81 篇  Ding, Shujiang       orcid:0000-0002-5683-0973
    71 篇  Su, Ya-Qiong         orcid:0000-0001-5581-5352
    66 篇  Zhang, Lei           orcid:0000-0003-2343-084X
    61 篇  Xu, Feng             orcid:0000-0003-4351-0222
    43 篇  Duan, Xin-Hua        orcid:0000-0002-1527-5934
    42 篇  Zhang, Mingzhen      orcid:0000-0002-4686-6526
    39 篇  Zheng, Yan-Zhen      orcid:0000-0003-4056-097X
    38 篇  Guo, Baolin          orcid:0000-0001-6756-1441
    36 篇  Zhou, Di             orcid:0000-0001-7411-4658
    36 篇  Lu, Guanghao         orcid:0000-0001-7829-7308


## 第九步：保存

- `data/interim/articles.csv`：文章宽表（多值字段用 `||` 分隔）。
- `data/interim/authors.csv`：作者长表（含 ORCID 与单位）。

> 用 `utf-8-sig` 保存，方便直接在 Excel 里打开；Python 读回时请写 `pd.read_csv(..., encoding='utf-8-sig')`。


In [13]:
interim_dir = os.path.join(os.getcwd(), '..', 'data', 'interim')
os.makedirs(interim_dir, exist_ok=True)

articles_path = os.path.join(interim_dir, 'articles.csv')
authors_path = os.path.join(interim_dir, 'authors.csv')

df.to_csv(articles_path, index=False, encoding='utf-8-sig')
df_authors.to_csv(authors_path, index=False, encoding='utf-8-sig')

print(f"已保存 {articles_path}")
print(f"    {len(df)} 行 × {df.shape[1]} 列")
print(f"已保存 {authors_path}")
print(f"    {len(df_authors)} 行 × {df_authors.shape[1]} 列")


已保存 D:\MEDscience_map_of_XJTU.wt\analysis\notebooks\..\data\interim\articles.csv
    29907 行 × 22 列
已保存 D:\MEDscience_map_of_XJTU.wt\analysis\notebooks\..\data\interim\authors.csv
    275392 行 × 6 列


## 小结

- 6 个 `.nbib` 文件共解析 **32,873** 条记录，按 `PMID` 去重后得到 **29,907** 篇文章。
- 产出：
  - `data/interim/articles.csv` —— 一篇文章一行；MeSH（`mesh`）与作者关键词（`keywords`）**分开保存**；`authors_key` 是按顺序拼好的作者身份键。
  - `data/interim/authors.csv` —— 作者长表，带 ORCID、单位和 `author_key`。
- 修复记录：
  - ✅ #01 不再经过 Zotero，直接解析 `.nbib`；MeSH 与作者关键词不再混淆。
  - ✅ #02 作者身份归一改为「ORCID 优先 → 全名 + 单位」，同名不同单位不再被合并（规则版，残余局限见第八步）。
- 下一步：新建 `02_*.ipynb`，基于 `articles.csv` / `authors.csv` 做关键词共现、作者合作、时间趋势等分析，并生成 VOSviewer 输入文件；同时用 `author_key` 检查仍有问题的作者个例。